In [1]:
!pip install qiskit qiskit-aer -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.3 MB/s eta 0:00:00


In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

fib = [1,1,2,3,5,8,13,21]
phi = 1.6180339887
gaps = [f/21*phi for f in fib]

def run_breathing():
    qc = QuantumCircuit(2,1)
    qc.h(0); qc.cx(0,1)
    for i,g in enumerate(gaps):
        amp = fib[i]/fib[i+1] if i < 7 else 0.618
        qc.rx(amp*np.pi*0.1, 0)
        qc.cx(0,1)
        qc.delay(int(g*100),0)
    qc.measure(0,0)
    return AerSimulator().run(qc, shots=1024).result().get_counts()

def run_fixed():
    qc = QuantumCircuit(2,1)
    qc.h(0); qc.cx(0,1)
    for _ in range(8):
        qc.rx(0.618*np.pi*0.1, 0)
        qc.cx(0,1)
        qc.delay(100,0)
    qc.measure(0,0)
    return AerSimulator().run(qc, shots=1024).result().get_counts()

print("BREATHING (fib flux):", run_breathing())
print("FIXED (regular):", run_fixed())

BREATHING (fib flux): {'1': 490, '0': 534}
FIXED (regular): {'0': 490, '1': 534}


In [3]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise = NoiseModel()
noise.add_all_qubit_quantum_error(depolarizing_error(0.05,1), ['rx','delay'])
noise.add_all_qubit_quantum_error(depolarizing_error(0.1,2), ['cx'])

def run_noisy(breathing=True):
    qc = QuantumCircuit(2,1)
    qc.h(0); qc.cx(0,1)
    seq = gaps if breathing else [0.618]*8
    for i,g in enumerate(seq):
        amp = fib[i]/fib[i+1] if breathing and i<7 else 0.618
        qc.rx(amp*np.pi*0.1, 0)
        qc.cx(0,1)
        qc.delay(int(g*100),0)
    qc.measure(0,0)
    return AerSimulator(noise_model=noise).run(qc, shots=1024).result().get_counts()

print("NOISY BREATHING:", run_noisy(True))
print("NOISY FIXED:", run_noisy(False))

NOISY BREATHING: {'1': 487, '0': 537}
NOISY FIXED: {'0': 523, '1': 501}
